# The LLM Architecture Blueprint: From Math to Code

This document breaks down the forward pass of a standard Autoregressive Transformer (like GPT). 

**Our Baseline Configuration:**
* **Sequence Length (`T` or `block_size`):** 1024 tokens
* **Embedding Dimension (`C` or `n_embd`):** 384
* **Number of Heads (`n_heads`):** 6
* **Head Dimension (`head_dim`):** 64 (because $384 / 6 = 64$)
* **Batch Size (`B`):** 1 (kept at 1 for simpler math)


---

## 1. Input Preparation: Embeddings & Positional Encoding

**What it is:** The process of converting raw text into numbers, and then adding spatial awareness to those numbers.
* **Token Embedding:** Maps an integer (token ID) to a dense vector of size 384.
* **Positional Encoding:** Adds a unique pattern of numbers to each token's vector based on its position (0 to 1023).

**Why we need it:** Unlike older models (RNNs) that read text sequentially word-by-word, Transformers read the entire 1024-token chunk simultaneously. Without positional encoding, the model would see "The dog bit the man" and "The man bit the dog" as the exact same bag of words.

**Advantages:** Enables massive parallel processing on GPUs since we don't have to wait for token $t$ to process token $t+1$.

**The Data Flow & Shapes:**
1.  **Raw Input:** `(1024)` -> 1D sequence of integers.
2.  **Token Embedding:** `(1024, 384)`
3.  **+ Positional Encoding:** `(1024, 384)` -> *This is the input $x$ that enters the Transformer Block.*

---

## 2. The Core Engine: Self-Attention

**What it is:** The mechanism that allows each token to look at every other token in the sequence to gather context. 



**Why we need it:** Words change meaning based on context. In the sentence "The bank of the river," "bank" means land. In "The bank on the corner," it means a financial institution. Self-attention allows the vector for "bank" to pull information from "river" or "corner" to update its own meaning.

**Advantages:** Creates a "global receptive field." A token at position 1000 can directly look at a token at position 1 in a single computational step.

**The Data Flow & Shapes:**
1.  **Create Q, K, V:** We multiply input $x$ by three learned weight matrices ($W_q, W_k, W_v$).
    * **Shape:** `(1024, 384)` each.
    * *Code Map:* `qkv = self.c_attn(x)` then `q, k, v = qkv.split(...)`
2.  **Raw Scores:** $Q \times K^T$
    * **Shape:** `(1024, 1024)` -> *An attention grid showing how every token relates to every other token.*
3.  **Scale:** Divide by $\sqrt{d_k}$ (which is 8 in our case).
    * **Why?** If dot products get too large, the subsequent Softmax function gets pushed into flat regions where gradients vanish (the model stops learning). Scaling keeps the variance stable.
4.  **Softmax:** Converts the grid into percentages that sum to 1.0 across rows.
5.  **Context Matrix:** Multiply probabilities by $V$.
    * **Shape:** `(1024, 384)`

---

## 3. Parallel Processing: Multi-Head Attention (MHA)

**What it is:** Instead of calculating one large attention grid using the full 384 dimensions, we slice the embeddings into 6 smaller chunks (heads) of 64 dimensions each.



**Why we need it:** A single word plays multiple roles in a sentence. It has a grammatical role (noun/verb), an emotional tone (positive/negative), and a semantic link (who it refers to). Splitting into "heads" allows the model to learn multiple different relationships simultaneously. 

**Advantages:** Massively improves the richness of the representations. Head 1 might learn to track pronouns, while Head 2 learns to track punctuation.

**The Data Flow & Shapes:**
1.  **Reshape & Transpose:** Slice the 384 dims into 6 heads of 64. 
    * **Shape:** `(1, 6, 1024, 64)`
    * *Code Map:* `q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)`
2.  **Parallel Attention:** The $Q \times K^T$ math happens 6 times at once.
    * **Shape:** `(1, 6, 1024, 1024)` -> *We now have six separate attention grids!*
3.  **Apply Values:** Multiply by $V$.
    * **Shape:** `(1, 6, 1024, 64)`
4.  **Recombine:** Concatenate the 6 heads back together.
    * **Shape:** `(1, 1024, 384)`
    * *Code Map:* `y.transpose(1, 2).contiguous().view(B, T, C)`

---

## 4. Autoregressive Enforcement: Causal Masking

**What it is:** A lower-triangular matrix of $1$s and $0$s used to overwrite future attention scores with $-\infty$.



**Why we need it:** During training, we feed the model the entire 1024-word sequence at once for GPU efficiency. However, the model's job is to predict the *next* word. If token #5 can look at token #6 in the attention grid, it's cheating.

**Advantages:** Allows us to train on thousands of words in parallel while strictly enforcing the rule that tokens can only gather context from the past and the present. 

**The Data Flow & Shapes:**
1.  **Mask Application:** Overlay the mask on the raw `(1024, 1024)` attention grid (per head).
    * *Code Map:* `attn_scores.masked_fill(self.mask[:,:, :T, :T] == 0, float('-inf'))`
2.  **Softmax Effect:** $e^{-\infty} = 0$. The probabilities of looking at any future token become exactly 0%.

---

## 5. The Reasoning Engine: Feed-Forward Network (FFN)

**What it is:** A standard multi-layer perceptron (MLP) applied to each token's 384-dimensional vector individually.

**Why we need it:** Attention just moves information *between* tokens. The FFN is where the model actually "thinks" about what that new information means. It acts as a memory bank and a pattern recognizer. 

**Advantages:** The non-linear activation function (like GELU) allows the model to map complex, non-linear representations of language that simple matrix multiplication cannot capture.

**The Data Flow & Shapes:**
1.  **Expansion Layer:** Usually scales up by 4x.
    * **Shape:** `(1024, 1536)`
2.  **Activation:** Apply GELU.
3.  **Contraction Layer:** Project back down.
    * **Shape:** `(1024, 384)`

---

## 6. Optimization: Standard Math vs. Flash Attention

In your PyTorch code, you implemented a switch between `flash-attention` and `own` (standard math). Here is why that switch matters:

**The Problem with Standard Math:**
Creating that `(1, 6, 1024, 1024)` attention grid takes up a massive amount of VRAM. If your `block_size` increases from 1024 to 8000, the grid size grows quadratically ($O(N^2)$), causing Out-Of-Memory (OOM) errors. Furthermore, moving this massive grid between the GPU's main memory (HBM) and its processing cores (SRAM) is incredibly slow.

**The Flash Attention Solution:**
Flash Attention is a hardware-aware algorithm. Instead of calculating the whole `1024x1024` grid and saving it to main memory, it computes the attention in small "blocks" directly on the ultra-fast SRAM chip, applies the softmax block-by-block, and only saves the final `(1024, 64)` output back to main memory.

* **Advantage:** It mathematically yields the *exact same result* as standard attention, but uses a fraction of the memory and is significantly faster. 
* *Code Map:* `F.scaled_dot_product_attention(..., is_causal=True)` automatically triggers the Flash Attention backend in PyTorch 2.0+.

***

This document covers the end-to-end journey of a tensor through the Transformer block. Would you like to dive into the architecture of the **KV Cache** next to see how this math changes during the generation phase (inference) to speed up response times?

In [1]:
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cpu' # torch.device("cuda" if torch.cuda.is_available() else "cpu")

## GELU (Gaussian Error Linear Unit)
The **GELU activation function** is defined as:

$$
\text{GELU}(x) = x \cdot \Phi(x)
$$

Where:

* ($\Phi(x)$) is the **cumulative distribution function (CDF)** of the standard normal distribution.




### Approximation (Tanh-based):

A commonly used fast approximation is:

$$
\text{GELU}(x) \approx 0.5x \left(1 + \tanh\left(\sqrt{\frac{2}{\pi}} \left(x + 0.044715 x^3 \right)\right)\right)
$$

---


* GELU can be seen as **stochastic regularization**:

  * Instead of hard gating like ReLU, it **weights inputs smoothly**.
* It keeps small negative values (unlike ReLU which zeros them).
* Acts like a **smooth version of ReLU**.

---

### Properties

* **Smooth and differentiable everywhere**
* Non-monotonic (slight curvature in negative region)
* Better gradient flow than ReLU
* Works well in deep architectures



In [ ]:
class AppGELU(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, x):
        out = 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))
        return out

In [ ]:
import matplotlib.pyplot as plt

relu, gelu = nn.ReLU(), AppGELU()

x = torch.linspace(-3, 3, 100)
y_gelu, y_relu = gelu(x), relu(x)

plt.figure(figsize=(8, 3))

for i, (y, lable) in enumerate(zip([y_gelu, y_relu], ['GELU', 'RELU']), 1):
    plt.subplot(1, 2, i)
    plt.plot(x, y)
    plt.title(f"{lable} activation function")
    plt.xlabel("x")
    plt.ylabel(f"{lable}(x)")
    plt.grid(True)

plt.tight_layout()
plt.show()

# GPT Implementation

In [3]:
@dataclass
class GPTConfig:
    block_size: int = 1024  # This is window size or context length
    vocab_size: int = 50304  # total number of tokens in tokeniser
    n_embd: int = 768 # 384       # Embedding dimmension size i.e each token is representation of this size
    n_layer: int = 12
    n_head: int = 12
    bias: bool = True
    embd_pdrop = 0.1
    dropout = 0.1
    attn_pdrop = 0.1
    resid_pdrop = 0.1
    attention_type = 'flash-attention' # 'flash-attention', 'without-flash-attention', 'manual'
    batch_size = 2

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Ensure embedding dimension is perfectly divisible by the number of heads
        assert config.n_embd % config.n_head == 0
        
        self.n_heads = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        
        # Read attention mode from config (defaulting to flash-attention)
        self.attention_mode = getattr(config, 'attention_type', 'flash-attention')

        # Q, K, V projections bundled into one linear layer for speed
        self.c_attn = nn.Linear(config.n_embd, config.n_embd * 3, bias=config.bias)
        # Output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        
        # Regularization
        self.dropout_p = config.dropout
        self.attn_dropout = nn.Dropout(config.attn_pdrop)
        self.resid_dropout = nn.Dropout(config.resid_pdrop)

        # Causal mask (only necessary for the 'own' implementation)
        # We use register_buffer so PyTorch moves it to the correct device automatically
        if self.attention_mode == 'manual':
            self.register_buffer(
                'mask', 
                torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size)
            )

    def forward(self, x):
        B, T, C = x.size() # Batch size, Sequence length, Embedding dimension

        # 1. Project to Q, K, V
        qkv = self.c_attn(x) # (B, T, n_embd * 3)
        q, k, v = qkv.split(self.n_embd, dim=2) # Split into three (B, T, n_embd) tensors

        # 2. Reshape and transpose for multi-head attention
        # Shape becomes: (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # 3. Handle dropout train/inference state for functional attention
        current_dropout_p = self.dropout_p if self.training else 0.0

        # 4. Compute Attention based on the selected mode
        if self.attention_mode == 'flash-attention':
            # PyTorch 2.0+ handles FlashAttention natively if is_causal=True and no mask is passed.
            y = F.scaled_dot_product_attention(
                q, k, v, 
                attn_mask=None, 
                dropout_p=current_dropout_p, 
                is_causal=True
            )
            
        elif self.attention_mode == 'without-flash-attention':
            # We explicitly force PyTorch's SDPA to disable the flash backend.
            # It will fall back to memory-efficient or standard math attention.
            with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=True, enable_mem_efficient=True):
                y = F.scaled_dot_product_attention(
                    q, k, v, 
                    attn_mask=None, 
                    dropout_p=current_dropout_p, 
                    is_causal=True
                )
        
        else:
            # 'own' implementation: Manual Math
            # Q @ K^T / sqrt(d)
            attn_scores = (q @ k.transpose(-2, -1)) * (1.0 / (self.head_dim ** 0.5))
            
            # Apply causal mask (block out future tokens)
            attn_scores = attn_scores.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
            
            # Softmax to get probabilities
            attn_scores = F.softmax(attn_scores, dim=-1)
            
            # Apply attention dropout (using the nn.Module so it respects model.eval())
            attn_scores = self.attn_dropout(attn_scores)

            # Multiply by Values
            y = attn_scores @ v # (B, n_heads, T, head_dim)
        
        # 5. Re-assemble multi-head outputs and apply final projection
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        
        # Residual dropout and output projection
        y = self.resid_dropout(self.c_proj(y))
        
        return y
        

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.c_fc = nn.Linear(config.n_embd , 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)

        return x


class Block(nn.Module):
    """ Transformer Block with self-attention and MLP Layer followed by Layer Normalization with residual connection. """
    def __init__(self, config):
        super().__init__()

        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT(nn.Module):
    
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd), # token embedding
            wpe = nn.Embedding(config.block_size, config.n_embd), # positional encoding
            emb_drop = nn.Dropout(config.embd_pdrop),
            h = nn.Sequential(*[Block(config) for _ in range(config.n_layer)]), # attention blocks
            ln_f = nn.LayerNorm(config.n_embd)
        ))

        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.transformer.wte.weight

    def forward(self, idx):
        B, T = idx.shape

        assert T <= self.config.block_size , f"input sequence of length `{T}` is greater than context length of `{self.config.block_size}`."
        
        tok_embeds = self.transformer.wte(idx)
        pos_embeds = self.transformer.wpe(torch.arange(T, device=idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.transformer.emb_drop(x)
        x = self.transformer.h(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        return logits

    @classmethod
    def from_pretrained(cls, model_type='gpt2'):
        from transformers import GPT2LMHeadModel

        # get the GPT-2 124M parameter model from huggingface transformer library
        download_dir = r"E:\code\DL\LLM-Lab\.model"
        model_hf = GPT2LMHeadModel.from_pretrained('gpt2', cache_dir=download_dir)
        sd_hf = model_hf.state_dict()
        sd_keys_hf = sd_hf.keys()

        # get the config for GPT-2 model
        config = GPTConfig
        
        # set the required parameters for GPT model
        config.vocab_size = 50257 
        config.block_size = 1024
        config.n_layer = 12
        config.n_head = 12

        # create the model 
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()

        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"

        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape, f"{sd_hf[k].shape[::-1]}, and {sd[k].shape}"
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape, f"{sd_hf[k].shape} and {sd[k].shape}"
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])
        
        return model
    
    def count_parameters(self, trainable_only: bool = False) -> int:
        """
        Count the number of parameters in this model.
        Args:
            trainable_only (bool): If True, counts only parameters with requires_grad=True
        Returns:
            int: Total number of parameters
        """
        if trainable_only:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        else:
            return sum(p.numel() for p in self.parameters())

## Text generation method

In [5]:
@torch.no_grad()
def generate(model, idx, max_tokens, temperature=1.0, top_k=None):
    model.eval()
    
    for _ in range(max_tokens):
        idx_cond = idx if idx.size(1) <= model.config.block_size else idx[:, -model.config.block_size:]
        
        logits = model(idx_cond)
        logits = logits[:, -1, :] / temperature

        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')

        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)

        idx = torch.cat((idx, idx_next), dim=1)

    return idx


def run_inference_pipeline(model, tokeniser, prompt, max_tokens=50, temperature=1.0, device='cpu'):
    input_ids = tokeniser.encode(prompt)  # list[int]
    idx = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0)  # (1, seq_len)

    # device = next(model.parameters()).device
    idx = idx.to(device)
    model = model.to(device)
    output_ids = generate(model, idx, max_tokens, temperature, 50)

    output_tokens = output_ids[0].tolist()
    output_text = tokeniser.decode(output_tokens)
    
    return output_text

In [6]:
import tiktoken

tokenizer = tiktoken.get_encoding('gpt2')

# Generate some sample

In [8]:
gpt_model = GPT.from_pretrained() # trained model

prompt = "Once upon a time in a small village "

run_inference_pipeline(gpt_model, tokenizer, prompt)

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 5b2dd5e5-f07f-4eda-9e85-5c582a165990)')' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/generation_config.json
Retrying in 1s [Retry 1/5].


'Once upon a time in a small village \xa0-a single house with its own fire - a little village - a whole village had gone to the trouble of setting fire to a couple of wooden houses. \xa0The only thing that helped was to let the people out through the doors - as'

### Data Loader

In [7]:
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

class GPTDataset(Dataset):
    """Contiguous-token sliding-window dataset for next-token prediction.
    Stores tokens in one contiguous `torch.LongTensor` and returns (input, target) pairs.
    """
    def __init__(self, txt_list, tokenizer, max_length, stride=1, add_eos=True):
        self.max_length = max_length
        self.stride = stride

        # Build flat token list, appending an EOS/sep token between documents if found
        token_ids = []

        for txt in txt_list:
            # ids = tokenizer.encode(txt + "<|endoftext|>", allowed_special={"<|endoftext|>"})
            ids = tokenizer.encode(txt)
            token_ids.extend(ids)

        if len(token_ids) <= max_length:
            raise ValueError(f"Corpus too small: total tokens={len(token_ids)} <= max_length={max_length}")

        # one contiguous tensor for memory locality
        tokens = torch.tensor(token_ids)
        num_blocks = len(tokens) // max_length

        self.x = tokens[:num_blocks * max_length].view(-1, max_length)
        self.y = tokens[1:num_blocks * max_length + 1].view(-1, max_length)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

def create_dataloader_v1(txt, tokenizer, batch_size=2, max_length=256, shuffle=True, drop_last=True, num_workers=0, pin_memory=True, stride=1):
    dataset = GPTDataset(txt, tokenizer, max_length, stride=stride, add_eos=True)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers, pin_memory=pin_memory)


c:\Users\Devendra\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
from datasets import load_dataset

# reading the file
ds = load_dataset(
        "parquet",
        data_files="E:\code\DL\LLM-Lab\data\small_tiny_stories\part-00000-221bb1a7-7aa9-4850-bf6f-a9fdb712fa71-c000.snappy.parquet",
    )

train_test_split = 0.9
n = int(ds['train'].num_rows * train_test_split)

train_ds = ds['train'][:n]['text']
val_ds = ds['train'][n:]['text']

In [ ]:
train = create_dataloader_v1(train_ds, tokenizer, batch_size=4, max_length=1024)

In [12]:
val = create_dataloader_v1(val_ds, tokenizer, batch_size=2, max_length=1024)

# Training and Model Evaluation

In [19]:
num_epoch = 1

In [24]:
@torch.no_grad()
def evaluate_loss(model, val_loader, train_loader):
    model.eval()
    total_val_loss = 0.0
    total_train_loss = 0.0

    # Validation loss
    for inputs, targets in val_loader:
        outputs = model(inputs)
        loss = F.cross_entropy(outputs.view(-1, outputs.size(-1)), targets.view(-1))
        total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    # Training loss (sample same number of batches as val_loader)
    train_iter = iter(train_loader)
    for _ in range(len(val_loader)):
        x, y = next(train_iter)
        outputs = model(x)
        loss = F.cross_entropy(outputs.view(-1, outputs.size(-1)), y.view(-1))
        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(val_loader)

    model.train()
    return avg_train_loss, avg_val_loss

def train(model, tokenizer, optimizer, train_loader, val_loader, num_epoch, device, eval_freq=100, start_token="Once " ):
    trained_losses, val_losses, trac_tokens_seen = [], [], []
    tokens_seen = 0
    steps = 0
    total_train_loss = 0

    model.train()

    for epoch in range(num_epoch):
        for x, y in train_loader:
            # set the gradients to zero
            optimizer.zero_grad()
            # get the output from model
            logits = model(x)
            # calculate cross entropy loss
            loss = F.cross_entropy(logits.view(-1, logits.size(-10)), y.flatten(-1))
            total_train_loss += loss.item()
            # accumulate gradients
            loss.backward()
            # update the weight parameters
            optimizer.step()
            tokens_seen += x.numel()
            step += 1

            if step % eval_freq == 0:
                train_loss, val_loss = evaluate_loss(model, val_loader)
                trained_losses.append(train_loss)
                val_losses.append(val_loss)
                trac_tokens_seen.append(tokens_seen)

                print(f"Epoch: {epoch} ,step {steps}: train loss {train_loss:.4f}, val loss {val_loss:.4f}, tokens_seen: {tokens_seen}")
                print("generated text : ")
                print(run_inference_pipeline(model, tokenizer, start_token))
    
    return trained_losses, val_losses, trac_tokens_seen
